# Day 3 — Chunking Strategies for RAG

Why chunking matters: LLMs have fixed context windows. The way you split your documents directly affects retrieval quality.
A poorly chunked document might split a product price across two chunks, or merge unrelated sections that confuse the retriever.
This notebook walks through four chunking strategies — fixed-size, recursive, character, and semantic — and compares their output on the same document.
Each strategy has a different trade-off between speed, boundary quality, and accuracy.

In [ ]:
import sys
sys.path.insert(0, '../src')
from day3.chunking import (
    fixed_size_chunk, recursive_chunk, character_chunk, semantic_chunk,
    extract_metadata_from_chunk, compare_strategies, SAMPLE_TEXT
)
print("Chunking module loaded.")
print(f"Sample text: {len(SAMPLE_TEXT)} characters")

## 1. Fixed-Size Chunking

Splits every 400 characters with 80-character overlap. Fast but ignores sentence boundaries.
A sentence like 'The price is ₹24,900' may be split into '₹24,' and '900 AirPods'.
Good for prototyping.

In [ ]:
chunks = fixed_size_chunk(SAMPLE_TEXT, chunk_size=400, overlap=80, source="demo")
print(f"Fixed-size: {len(chunks)} chunks")
for i, c in enumerate(chunks[:3]):
    print(f"\nChunk {i}: {c.word_count} words | chars [{c.start_char}:{c.end_char}]")
    print(f"  '{c.text[:120]}...'")
print(f"\nMetadata example: {chunks[0].metadata}")

## 2. Recursive Chunking (LangChain-style)

Tries separators in order: \n\n → \n → '. ' → ' '. Preserves paragraphs and sentences wherever possible.
This is the best general-purpose strategy for product catalogs, support docs, and reports.

In [ ]:
recursive = recursive_chunk(SAMPLE_TEXT, chunk_size=400, overlap=80, source="demo")
print(f"Recursive: {len(recursive)} chunks")
for i, c in enumerate(recursive[:3]):
    print(f"\nChunk {i}: {c.word_count} words")
    print(f"  '{c.text[:120]}...'")

## 3. Character Chunking

Splits on a single separator (\n\n by default = paragraph breaks). Each chunk is one complete paragraph.
Best for FAQ pages, product catalogs, and markdown docs where each paragraph is a natural retrieval unit.

In [ ]:
char_chunks = character_chunk(SAMPLE_TEXT, separator="\n\n", source="demo")
print(f"Character (paragraph): {len(char_chunks)} chunks")
for i, c in enumerate(char_chunks[:3]):
    print(f"\nChunk {i}: {c.word_count} words")
    print(f"  '{c.text[:120]}...'")

## 4. Semantic Chunking

Groups sentences by embedding similarity. A topic change (cosine similarity drops below threshold) creates a new chunk.
Most accurate but requires an embedding model. Falls back to recursive chunking if no encoder is provided.

In [ ]:
# Without encoder (fallback to recursive — no model download needed)
semantic_fallback = semantic_chunk(SAMPLE_TEXT, encoder=None, source="demo")
print(f"Semantic (fallback): {len(semantic_fallback)} chunks")
print("Note: Pass encoder=SentenceTransformer().encode for real semantic chunking")
print("      when running with a GPU or allowing model download")

# With real encoder (only if sentence-transformers available)
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    import numpy as np
    def encoder_fn(texts):
        return model.encode(texts, convert_to_numpy=True)
    real_semantic = semantic_chunk(SAMPLE_TEXT, encoder=encoder_fn, similarity_threshold=0.7)
    print(f"\nSemantic (real encoder): {len(real_semantic)} chunks")
    for i, c in enumerate(real_semantic[:2]):
        print(f"  Chunk {i}: {c.metadata.get('num_sentences', '?')} sentences | {c.word_count} words")
except Exception as e:
    print(f"\n(Skipping real encoder: {e})")

## 5. Metadata Extraction

When indexing into ChromaDB or Databricks Vector Search, you can attach metadata to each chunk.
This enables pre-filtering: 'only search laptop documents' before vector search, cutting retrieval time dramatically.

In [ ]:
meta = extract_metadata_from_chunk(
    SAMPLE_TEXT[:600], source_file="product_catalog.txt", chunk_index=0
)
print("Extracted metadata:")
for k, v in meta.items():
    print(f"  {k:25s}: {v}")

## 6. Strategy Comparison

Run all four strategies on the same document and compare num_chunks, avg_words, min/max.
Use this to decide which strategy to pick for your document type before committing to production.

In [ ]:
comparison = compare_strategies(SAMPLE_TEXT)
print(f"{'Strategy':<12} {'Chunks':>6} {'AvgWords':>9} {'Min':>5} {'Max':>5}")
print("-" * 45)
for name, stats in comparison.items():
    print(f"{name:<12} {stats['num_chunks']:>6} {stats['avg_words']:>9.1f} "
          f"{stats['min_words']:>5} {stats['max_words']:>5}")

print("\nSample first chunk per strategy:")
for name, stats in comparison.items():
    print(f"  [{name}] '{stats['sample_chunk'][:80]}...'")

## Databricks Bridge

In production Databricks, chunking happens in a Bronze→Silver notebook.
Chunks are stored in a Delta table with metadata columns, then synced to Databricks Vector Search.

In [ ]:
print("""
DATABRICKS EQUIVALENT:
  Bronze table: dev_agents.naval.raw_products  (full product text)
  Silver table: dev_agents.naval.product_chunks (one row per chunk)
  
  Schema: chunk_id, text, strategy, word_count, has_price, category, source_file
  
  Delta table creation:
    spark.createDataFrame(chunks_df).write.format("delta")
         .saveAsTable("dev_agents.naval.product_chunks")
  
  Then sync to Databricks Vector Search:
    vs_client.create_delta_sync_index(
        endpoint_name="one-env-shared-endpoint-0",
        source_table="dev_agents.naval.product_chunks",
        primary_key="chunk_id",
        embedding_source_column="text",
        embedding_model_endpoint_name="databricks-bge-large-en"
    )
""")